# 05 — Pseudo-Labeling (Self-Training)

Iteratively fine-tunes `utils.config.CLASSIFIER_MODEL_NAME` on the labeled
pool, predicts on the unlabeled pool, and absorbs high-confidence
predictions each round. Uses `CLASSIFIER_SAMPLE_SIZE` (much smaller than
the embedding tasks' `SAMPLE_SIZE`) since fine-tuning is far more
CPU-expensive than frozen embedding — this is still the most expensive
notebook in the plan. The full-data final run should be kicked off
unattended and budgeted in hours, not minutes.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json

import pandas as pd

from utils import config
from utils.data import stratified_sample
from utils.metrics import evaluate_label_quality, evaluate_semisupervised
from utils.modeling import get_predictions, pseudo_label_loop

In [2]:
labeled_df = pd.read_parquet(config.PROCESSED_DIR / "labeled.parquet")
unlabeled_df = pd.read_parquet(config.PROCESSED_DIR / "unlabeled.parquet")
test_clean = pd.read_parquet(config.PROCESSED_DIR / "test_clean.parquet")

labeled_sample = stratified_sample(labeled_df, config.CLASSIFIER_SAMPLE_SIZE, seed=config.SEED)
unlabeled_sample = stratified_sample(unlabeled_df, config.CLASSIFIER_SAMPLE_SIZE, seed=config.SEED)

overlap = set(unlabeled_sample["text"]) & set(test_clean["text"])
assert len(overlap) == 0, f"{len(overlap)} rows leaked between train pool and test set"
print(f"Labeled sample: {len(labeled_sample)} | Unlabeled sample: {len(unlabeled_sample)} | Test: {len(test_clean)}")
print("No train/test text overlap confirmed.")

Labeled sample: 152 | Unlabeled sample: 150 | Test: 7600
No train/test text overlap confirmed.


In [3]:
final_model, final_tokenizer, current_labeled, history = pseudo_label_loop(
    labeled_sample, unlabeled_sample,
    model_name=config.CLASSIFIER_MODEL_NAME, n_iterations=3,
    confidence_threshold=0.90, epochs=3)

for h in history:
    print(h)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


C:\Users\ACER\OneDrive\Documents\final-project\.worktrees\autolabel-notebooks\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


C:\Users\ACER\OneDrive\Documents\final-project\.worktrees\autolabel-notebooks\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


{'iteration': 0, 'new_labels': 0, 'labeled_size': 152, 'unlabeled_size': 150}
{'iteration': 1, 'new_labels': 0, 'labeled_size': 152, 'unlabeled_size': 150}
{'iteration': 2, 'new_labels': 0, 'labeled_size': 152, 'unlabeled_size': 150}


In [4]:
pseudo_only = current_labeled.iloc[len(labeled_sample):]
merged = pseudo_only.merge(unlabeled_sample[["text", "true_label"]], on="text", how="left")

label_quality = evaluate_label_quality(
    true_labels=merged["true_label"].to_numpy(),
    pseudo_labels=merged["label"].to_numpy())
print("Pseudo-label quality:", label_quality)

Pseudo-label quality:

 {'Label Accuracy': 0.0, 'Label Macro F1': 0.0, 'Coverage': 0.0}


In [5]:
test_probs = get_predictions(final_model, final_tokenizer, test_clean["text"].tolist())
test_preds = test_probs.argmax(axis=1)

semisup_results, report, cm = evaluate_semisupervised(
    test_clean["label"].to_numpy(), test_preds, config.CLASS_NAMES,
    save_path=config.RESULTS_DIR / "confusion_matrix_pseudo_labeling.png")
print(report)

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(config.RESULTS_DIR / "metrics_pseudo_labeling.json", "w") as f:
    json.dump({"test_metrics": semisup_results, "label_quality": label_quality, "history": history},
               f, indent=2)
print("Saved pseudo-labeling results.")

              precision    recall  f1-score   support

       World       0.77      0.90      0.83      1900
      Sports       0.95      0.98      0.96      1900
    Business       0.76      0.80      0.78      1900
    Sci/Tech       0.85      0.64      0.73      1900

    accuracy                           0.83      7600
   macro avg       0.83      0.83      0.83      7600
weighted avg       0.83      0.83      0.83      7600

Saved pseudo-labeling results.
